# Problem 1  
Implement a feed-forward neural network in PyTorch.
1. Construct a synthetic regression dataset with a train and test dataset, where a
one-layer network does not show good performance on the test dataset, while a
network with more layers shows good performance.
2. Can you propose a regularization term that reduces the performance gap between
the test and train datasets for the larger network? Show empirically that your
regularizer works.

In [1]:
import torch
import torch.nn as nn

class SyntheticRegressionDataset(torch.utils.data.Dataset):
    def __init__(self, num_samples, input_size):
        self.X = torch.randn(num_samples, input_size)
        # self.y = torch.sum(self.X, dim=1) * 0.4 + torch.randn(num_samples) * 0.1 # Add some noise
        self.y = torch.sin(torch.sum(self.X, dim=1)) + torch.randn(num_samples) * 0.2 # Add some noise

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

/home/vaishnavahari/26ss/(VU) Deep Learning for Structured Data/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:

class SimpleNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, output_size)

    def forward(self, x):
        out = self.fc1(x)
        return out

class MultiLayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiLayerNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = torch.relu(self.fc1(x))
        out = self.fc2(out)
        return out

In [3]:
torch.manual_seed(0)
input_size = 10
output_size = 1

dataset = SyntheticRegressionDataset(1000, input_size)
# test_dataset = SyntheticRegressionDataset(200, input_size)
# split dataset into train and test
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

one_layer_model = SimpleNN(input_size, output_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(one_layer_model.parameters(), lr=0.01)

NUM_EPOCHS = 500

In [4]:
# Train the one-layer model
for epoch in range(NUM_EPOCHS):
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = one_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0) 
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {train_loss / len(train_loader):.4f}')
# Evaluate the one-layer model
one_layer_model.eval()
with torch.no_grad():
    test_loss = 0
    for X_batch, y_batch in test_loader:
        outputs = one_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        test_loss += loss.item() * X_batch.size(0)
print(f'One-layer model test loss: {test_loss / len(test_loader):.4f}')
one_layer_train_loss = train_loss / len(train_loader)
one_layer_test_loss = test_loss / len(test_loader)

Epoch [1/500], Loss: 23.3566
Epoch [2/500], Loss: 17.1917
Epoch [3/500], Loss: 16.8952
Epoch [4/500], Loss: 16.8783
Epoch [5/500], Loss: 16.9843
Epoch [6/500], Loss: 17.0962
Epoch [7/500], Loss: 16.9151
Epoch [8/500], Loss: 16.9108
Epoch [9/500], Loss: 16.8750
Epoch [10/500], Loss: 16.9453
Epoch [11/500], Loss: 16.9011
Epoch [12/500], Loss: 16.9858
Epoch [13/500], Loss: 17.0572
Epoch [14/500], Loss: 17.0403
Epoch [15/500], Loss: 16.9464
Epoch [16/500], Loss: 16.9940
Epoch [17/500], Loss: 17.1387
Epoch [18/500], Loss: 16.9440
Epoch [19/500], Loss: 17.0731
Epoch [20/500], Loss: 16.9422
Epoch [21/500], Loss: 16.9855
Epoch [22/500], Loss: 16.9762
Epoch [23/500], Loss: 17.0349
Epoch [24/500], Loss: 16.9311
Epoch [25/500], Loss: 16.9665
Epoch [26/500], Loss: 17.0111
Epoch [27/500], Loss: 17.0122
Epoch [28/500], Loss: 16.9502
Epoch [29/500], Loss: 16.9022
Epoch [30/500], Loss: 16.9612
Epoch [31/500], Loss: 16.9509
Epoch [32/500], Loss: 16.9190
Epoch [33/500], Loss: 16.9600
Epoch [34/500], Los

In [5]:
# Train multi-layer model
hidden_size = 20
multi_layer_model = MultiLayerNN(input_size, hidden_size, output_size)
optimizer = torch.optim.Adam(multi_layer_model.parameters(), lr=0.01)


In [6]:
for epoch in range(NUM_EPOCHS):
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {train_loss / len(train_loader):.4f}')
# Evaluate the multi-layer model
multi_layer_model.eval()
with torch.no_grad():
    test_loss = 0
    for X_batch, y_batch in test_loader:
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        test_loss += loss.item() * X_batch.size(0)
print(f'Multi-layer model test loss: {test_loss / len(test_loader):.4f}')
multi_layer_train_loss = train_loss / len(train_loader)
multi_layer_test_loss = test_loss / len(test_loader)

Epoch [1/500], Loss: 17.9806
Epoch [2/500], Loss: 17.0974
Epoch [3/500], Loss: 16.8079
Epoch [4/500], Loss: 16.5380
Epoch [5/500], Loss: 16.4141
Epoch [6/500], Loss: 16.4074
Epoch [7/500], Loss: 16.2053
Epoch [8/500], Loss: 16.2765
Epoch [9/500], Loss: 15.9806
Epoch [10/500], Loss: 16.0154
Epoch [11/500], Loss: 15.8053
Epoch [12/500], Loss: 15.5615
Epoch [13/500], Loss: 15.4202
Epoch [14/500], Loss: 15.5083
Epoch [15/500], Loss: 15.5534
Epoch [16/500], Loss: 15.3043
Epoch [17/500], Loss: 15.3519
Epoch [18/500], Loss: 14.8570
Epoch [19/500], Loss: 14.7034
Epoch [20/500], Loss: 14.0743
Epoch [21/500], Loss: 13.7380
Epoch [22/500], Loss: 13.2597
Epoch [23/500], Loss: 12.2158
Epoch [24/500], Loss: 11.4577
Epoch [25/500], Loss: 10.6664
Epoch [26/500], Loss: 10.3277
Epoch [27/500], Loss: 9.6524
Epoch [28/500], Loss: 9.7621
Epoch [29/500], Loss: 9.3816
Epoch [30/500], Loss: 9.4036
Epoch [31/500], Loss: 8.6973
Epoch [32/500], Loss: 8.4491
Epoch [33/500], Loss: 8.2772
Epoch [34/500], Loss: 8.78

In [7]:
print(f'One-layer model train loss: {one_layer_train_loss:.4f}, test loss: {one_layer_test_loss:.4f}')
print(f'Multi-layer model train loss: {multi_layer_train_loss:.4f}, test loss: {multi_layer_test_loss:.4f}')

One-layer model train loss: 17.0110, test loss: 15.6243
Multi-layer model train loss: 1.3003, test loss: 4.7986


Now with L2 regularization:

In [8]:
# Regularization L2
l2_lambda = 0.0001
multi_layer_model = MultiLayerNN(input_size, hidden_size, output_size)
optimizer = torch.optim.Adam(multi_layer_model.parameters(), lr=0.01, weight_decay=l2_lambda)


In [9]:
for epoch in range(NUM_EPOCHS):
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {train_loss / len(train_loader):.4f}')
multi_layer_model.eval()
with torch.no_grad():
    test_loss = 0
    for X_batch, y_batch in test_loader:
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        test_loss += loss.item() * X_batch.size(0)
print(f'Multi-layer model test loss: {test_loss / len(test_loader):.4f}')
multi_layer_regularized_train_loss = train_loss / len(train_loader)
multi_layer_regularized_test_loss = test_loss / len(test_loader)

Epoch [1/500], Loss: 17.7210
Epoch [2/500], Loss: 17.0754
Epoch [3/500], Loss: 16.7207
Epoch [4/500], Loss: 16.5178
Epoch [5/500], Loss: 16.3680
Epoch [6/500], Loss: 16.2418
Epoch [7/500], Loss: 16.4118
Epoch [8/500], Loss: 15.9732
Epoch [9/500], Loss: 16.0440
Epoch [10/500], Loss: 15.8614
Epoch [11/500], Loss: 15.7509
Epoch [12/500], Loss: 15.7333
Epoch [13/500], Loss: 15.5892
Epoch [14/500], Loss: 15.7841
Epoch [15/500], Loss: 15.7157
Epoch [16/500], Loss: 15.2041
Epoch [17/500], Loss: 15.3495
Epoch [18/500], Loss: 15.3995
Epoch [19/500], Loss: 15.0712
Epoch [20/500], Loss: 15.1286
Epoch [21/500], Loss: 15.1597
Epoch [22/500], Loss: 14.8997
Epoch [23/500], Loss: 14.9521
Epoch [24/500], Loss: 14.9389
Epoch [25/500], Loss: 14.8881
Epoch [26/500], Loss: 14.9065
Epoch [27/500], Loss: 14.7707
Epoch [28/500], Loss: 14.7657
Epoch [29/500], Loss: 14.4128
Epoch [30/500], Loss: 14.3590
Epoch [31/500], Loss: 14.4928
Epoch [32/500], Loss: 14.6321
Epoch [33/500], Loss: 14.4581
Epoch [34/500], Los

In [10]:
print(f'Multi-layer model with L2 regularization train loss: {multi_layer_regularized_train_loss:.4f}, test loss: {multi_layer_regularized_test_loss:.4f}')
print(f'Multi-layer model without regularization train loss: {multi_layer_train_loss:.4f}, test loss: {multi_layer_test_loss:.4f}')

Multi-layer model with L2 regularization train loss: 1.9108, test loss: 4.4801
Multi-layer model without regularization train loss: 1.3003, test loss: 4.7986


Test loss is lower with regularization, even with the penalty on the weights. If trained further, the difference is more visible.

In [11]:
for epoch in range(450):
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    print(f'Epoch [{epoch+1}/450], Loss: {train_loss / len(train_loader):.4f}')
multi_layer_model.eval()
with torch.no_grad():
    test_loss = 0
    for X_batch, y_batch in test_loader:
        outputs = multi_layer_model(X_batch)
        loss = criterion(outputs.squeeze(), y_batch)
        test_loss += loss.item() * X_batch.size(0)
print(f'Multi-layer model test loss: {test_loss / len(test_loader):.4f}')
multi_layer_regularized_train_loss = train_loss / len(train_loader)
multi_layer_regularized_test_loss = test_loss / len(test_loader)

Epoch [1/450], Loss: 2.0182
Epoch [2/450], Loss: 1.8769
Epoch [3/450], Loss: 1.8658
Epoch [4/450], Loss: 1.8648
Epoch [5/450], Loss: 1.9586
Epoch [6/450], Loss: 2.0106
Epoch [7/450], Loss: 2.0493
Epoch [8/450], Loss: 1.9525
Epoch [9/450], Loss: 1.8969
Epoch [10/450], Loss: 2.0597
Epoch [11/450], Loss: 1.8863
Epoch [12/450], Loss: 1.8091
Epoch [13/450], Loss: 1.9961
Epoch [14/450], Loss: 2.1160
Epoch [15/450], Loss: 1.8458
Epoch [16/450], Loss: 1.7602
Epoch [17/450], Loss: 1.8933
Epoch [18/450], Loss: 1.6909
Epoch [19/450], Loss: 1.7687
Epoch [20/450], Loss: 1.6807
Epoch [21/450], Loss: 1.7673
Epoch [22/450], Loss: 1.8823
Epoch [23/450], Loss: 1.8637
Epoch [24/450], Loss: 1.8344
Epoch [25/450], Loss: 1.8561
Epoch [26/450], Loss: 1.7369
Epoch [27/450], Loss: 1.7483
Epoch [28/450], Loss: 1.7503
Epoch [29/450], Loss: 1.8273
Epoch [30/450], Loss: 1.7659
Epoch [31/450], Loss: 1.7696
Epoch [32/450], Loss: 1.7996
Epoch [33/450], Loss: 1.7653
Epoch [34/450], Loss: 1.8452
Epoch [35/450], Loss: 1

# Problem 2  
Derive a hypothesis class H and a data distribution P such that there exists a hypothesis
h ∈ H for which the training error is zero, but the expected test error is no better than
random guessing.
1. Define a finite hypothesis class H with at least two hypotheses and describe a data
distribution P on (X , Y).
2. Construct a small training set drawn from P where one hypothesis hoverfit fits the
training data perfectly, achieving zero training error.
3. Explain why, under the chosen P, this hypothesis does no better than random
guessing on unseen test data.

#### Solution
Let $P$ be a distribution on $(X, Y)$,  
where $X \in \mathbb{R}^2$ and $Y \in \{0, 1\}$ is a binary label.

$X = (X_1, X_2)$ there exists a boundary in $X_{1,boundary}$ and $X_{2,boundary}$ such that:
- If $X_1 \ge X_{1,boundary}$ and $X_2 \ge X_{2,boundary}$, then $Y = 0$.
- If $X_1 \le X_{1,boundary}$ and $X_2 \le X_{2,boundary}$, then $Y = 0$.
- For all other combinations, $Y = 1$.

Then, 
1. Hypothesis class $H$ may contain:
- $h_1(X) = 0$ if $X_1 \ge X_{1,boundary}$ else $1$
- $h_2(X) = 0$ if $X_1 \le X_{1,boundary}$ 

2. Let the training set $S_N$ be defined as:  
 $S_{N, small region} = \{((X_1, X_2), Y) |  X_2 < X_{2,boundary}\}$

then, $h_2$ fits perfectly.

3. However, if a better training set is constructed as:  
$S_{N, large region} = \{((X_1, X_2), Y) |  X_{1,boundary} - \delta < X_1 < X_{1,boundary} + \delta, X_{2,boundary} - \delta < X_2 < X_{2,boundary} + \delta\}$


$h_2$ or $h_1$ will perform no better than random.



# Problem 3:

Need to show that   
$$
R(\hat{h}_n) \le R(h^*) + 2\epsilon
$$
or equivalently,
$$
R(\hat{h}_n) - R(h^*) \le 2\epsilon
$$


Start with LHS, and add and subtract $R(\hat{h}_n)$ and $\hat{R}(h^*)$:
$$
R(\hat{h}_n) - R(h^*) = [R(\hat{h}_n) - \hat{R}_n(\hat{h}_n)] + [\hat{R}_n(\hat{h}_n) - \hat{R}_n(h^*)] + [\hat{R}_n(h^*) - R(h^*)]
$$

On the RHS, 
for term 1, we can use Uniform Convergence for Finite Hypothesis Class Theorem for $h = \hat{h}_n$. And get $R(\hat{h}_n) - \hat{R}_n(\hat{h}_n) \le \epsilon $.  

and term 2, we can use the fact that $\hat{h}_n$ gives the minimum of $\hat{R}_n(h)$ than any other $h$, so $\hat{R}_n(\hat{h}_n) \le \hat{R}_n(h^*)$ or $\hat{R}_n(\hat{h}_n) - \hat{R}_n(h^*) \le 0$.  

and for term 3, again applying Uniform Convergence for Finite Hypothesis Class Theorem for $h = h^*$, we get $\hat{R}_n(h^*) - R(h^*) \le \epsilon$.

Finally, 
$$ R(\hat{h}_n) - R(h^*) \le \epsilon + 0 + \epsilon $$
     
$$ R(\hat{h}_n) - R(h^*) \le 2\epsilon $$
    